# B2-023-generative-models-diffusion — Practice p17 — Solution

**Type:** integrative · **Difficulty:** core · **Concepts:** generative-adversarial-network

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

## Solution

**Plan.** Build the pinned split readers, the two networks, the p06/p07 losses and helpers, and the p08 `mode_coverage`.
Seed, build $G$ then $D$, build the two Adam optimizers, and run `train_gan` for exactly 600 alternating steps on the 256 training rows, with all training noise from one dedicated generator seeded once inside `train_gan`.
For the duration of the training call only, `D.forward` is wrapped by a recorder that keeps a copy of every row the discriminator receives (real rows, and the generated rows of both half-steps); the original `forward` is restored before any evaluation.
The certificate then hashes the recorded rows with `generative_data.row_sha256` (each distinct row once) and checks them against the held-out and training parts of `ROW_SHA256["mixture2d"]`.

In [ ]:
import copy

DATASET = "mixture2d"


def train_rows():
    return generative_data.train_tensor(DATASET)


def heldout_rows():
    return generative_data.heldout_tensor(DATASET)


def make_generator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 2))


def make_discriminator():
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))


def _check_logits(*tensors):
    for x in tensors:
        if not isinstance(x, torch.Tensor) or x.dim() != 2 or x.shape[-1] != 1:
            raise ValueError("logits must be 2-D tensors with last dimension 1")
    if len({x.dtype for x in tensors}) != 1:
        raise ValueError("logits must share one dtype")


def discriminator_loss(real_logits, fake_logits):
    _check_logits(real_logits, fake_logits)
    return (F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
            + F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits)))


def generator_loss(fake_logits):
    _check_logits(fake_logits)
    return F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))


def d_step(G, D, opt_d, real, z):
    opt_d.zero_grad(set_to_none=True)
    real_logits = D(real)
    fake_logits = D(G(z).detach())
    loss = discriminator_loss(real_logits, fake_logits)
    loss.backward()
    opt_d.step()
    return loss.item()


def g_step(G, D, opt_g, z):
    opt_g.zero_grad(set_to_none=True)
    loss = generator_loss(D(G(z)))
    loss.backward()
    opt_g.step()
    return loss.item()


def gan_step(G, D, opt_g, opt_d, real, z):
    d_loss = d_step(G, D, opt_d, real, z)
    g_loss = g_step(G, D, opt_g, z)
    return (d_loss, g_loss)


def mode_coverage(samples, centers, radius):
    if not isinstance(samples, torch.Tensor) or not isinstance(centers, torch.Tensor):
        raise ValueError("samples and centers must be tensors")
    if samples.dim() != 2 or centers.dim() != 2 or samples.shape[1] != centers.shape[1]:
        raise ValueError("samples and centers must be 2-D with equal last dimensions")
    if samples.shape[0] == 0:
        raise ValueError("need at least one sample")
    if not isinstance(radius, (int, float)) or isinstance(radius, bool) or not radius > 0:
        raise ValueError("radius must be positive")
    d2 = ((samples[:, None, :] - centers[None, :, :]) ** 2).sum(dim=2)
    nearest = d2.argmin(dim=1)
    inside = d2.gather(1, nearest[:, None]).squeeze(1) <= radius ** 2
    counts = torch.bincount(nearest[inside], minlength=centers.shape[0]).to(torch.int64)
    need = math.ceil(samples.shape[0] / (8 * centers.shape[0]))
    return counts, int((counts >= need).sum().item())


def train_gan(G, D, opt_g, opt_d, batch):
    noise_generator = torch.Generator().manual_seed(SEED)
    trace = []
    for _ in range(600):
        z = torch.randn(batch.shape[0], 2, generator=noise_generator)
        trace.append(gan_step(G, D, opt_g, opt_d, batch, z))
    return trace


def split_hash_sets(name):
    hashes = generative_data.ROW_SHA256[name]
    return ({hashes[i] for i in generative_data.TRAIN_IDS[name]},
            {hashes[i] for i in generative_data.HELDOUT_IDS[name]})


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)


def hash_recorded(chunks):
    """Hash every distinct recorded row once with the loader's canonical row hash."""
    rows = torch.unique(torch.cat(chunks), dim=0)
    return {generative_data.row_sha256(row) for row in rows}, rows.shape[0]

### Training with the clean-row seam recorded

In [ ]:
torch.manual_seed(SEED)
G = make_generator()
D = make_discriminator()
opt_g = torch.optim.Adam(G.parameters(), lr=5e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
opt_d = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
G_init, D_init = copy.deepcopy(G), copy.deepcopy(D)

recorded = []
original_forward = D.forward


def recording_forward(x):
    recorded.append(x.detach().clone())
    return original_forward(x)


D.forward = recording_forward
try:
    trace = train_gan(G, D, opt_g, opt_d, train_rows())
finally:
    del D.forward                       # restore the class's original forward
forward_restored = "forward" not in vars(D) and D.forward == original_forward
recorded_calls = len(recorded)
recorded_hashes, recorded_unique = hash_recorded(recorded)

with torch.no_grad():
    samples = G(torch.randn(512, 2, generator=torch.Generator().manual_seed(SEED + 2)))
    counts, covered = mode_coverage(samples, generative_data.mixture_centers(), 0.6)
    p_train = torch.sigmoid(D(train_rows())).mean().item()
    p_held = torch.sigmoid(D(heldout_rows())).mean().item()

d_tail = sum(d for d, _ in trace[-50:]) / 50
g_max = max(g for _, g in trace)
g_changed = any(not torch.equal(a, b) for a, b in zip(G.parameters(), G_init.parameters()))
d_changed = any(not torch.equal(a, b) for a, b in zip(D.parameters(), D_init.parameters()))
windows = [tuple(round(sum(row[i] for row in trace[a:a + 50]) / 50, 3) for i in range(2)) for a in range(0, 600, 100)]
print("50-step window means (D, G) every 100 steps:", windows)
print(f"last-50 mean D loss {d_tail:.4f} (2 log 2 = {2 * math.log(2):.4f}); max G loss {g_max:.4f}")
print("coverage counts:", counts.tolist(), " covered:", covered)
print(f"mean D(train) {p_train:.4f}  mean D(held-out) {p_held:.4f}  gap {abs(p_train - p_held):.4f}")
print(f"recorded {recorded_calls} forward calls, {recorded_unique} distinct rows")

### Interpretation

The frozen-seed run gives counts `[0, 0, 181, 149]` for the 512 fixed-noise samples: components $(-1,-1)$ and $(1,-1)$ are covered (both far above the threshold $\lceil512/32\rceil=16$), components $(1,1)$ and $(-1,1)$ receive no counted sample, and $330/512\approx64\%$ of the samples lie within $0.6$ of a centre.
A last-50 discriminator loss near $2\log2$ is what a perfect generator would produce, but it is equally what a weak or recently fooled discriminator produces, so (B) shows only that the game is balanced, not that the generator reproduces the data.
With 600 steps of a two-player game on a four-blob mixture, a tiny MLP generator typically covers only some blobs at any moment, so "at least 2 of 4 covered modes" is the honest bar that the frozen-seed run supports; it rules out complete collapse onto one blob but explicitly admits partial mode collapse, which the count vector makes visible.
(E) compares training and held-out rows because a discriminator that had memorized the specific training rows would score them as "real" more confidently than fresh rows from the same distribution; the training score alone cannot separate "the discriminator learned the distribution" from "it memorized these 256 points".

### Answer check

In [ ]:
assert train_rows().shape == (256, 2) and heldout_rows().shape == (64, 2)
assert train_rows().dtype == torch.float32 and heldout_rows().dtype == torch.float32
# (A)
assert len(trace) == 600
assert all(isinstance(d, float) and isinstance(g, float) and math.isfinite(d) and math.isfinite(g) for d, g in trace)
# (B)
assert abs(d_tail - 2 * math.log(2)) <= 0.95
# (C)
assert all(math.isfinite(g) for _, g in trace) and g_max <= 5.0
# (D)
assert math.ceil(512 / 32) == 16 and covered >= 2
assert counts.dtype == torch.int64 and tuple(counts.shape) == (4,)
# (E)
assert abs(p_train - p_held) <= 0.10
# (F)
assert g_changed and d_changed
assert tuple(samples.shape) == (512, 2) and samples.dtype == torch.float32
# (G)
assert forward_restored
assert recorded_calls == 600 * 3
assert not (recorded_hashes & HELDOUT_HASHES)
assert TRAIN_HASHES <= recorded_hashes